# 07 — Mandelbrot Deep Dive

**Winner: Haskell** (~0.85 J) — **30.7× ahead of C++ (2nd, ~26 J)**. The largest margin of any benchmark.

Notable patterns:
- **Haskell's lead is extraordinary**: 0.85 J vs C++'s 26 J and C's 35 J. This is the most extreme result in the entire dataset.
- The Haskell `mandelbrot` implementation uses **LLVM-backed GHC** with aggressive SIMD vectorisation that produces highly optimised floating-point inner loops — effectively auto-vectorising the complex-number iteration.
- **Rust (3rd, ~34 J) and C (4th, ~35 J)** are neck-and-neck and close to C++.
- **Perl is worst** (~12 065 J, **14 258× worse than Haskell**) — the largest absolute gap in the dataset.
- Python (~7 018 J) and Ruby (~4 527 J) also suffer enormously on this floating-point intensive benchmark.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

BENCHMARK = 'mandelbrot'
RUNS_CSV  = Path('../results/results_clean_runs.csv')
MEAN_CSV  = Path('../results/results_clean.csv')
FIGS_DIR  = Path('../results/figs/mandelbrot')
FIGS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

def save(name): plt.savefig(FIGS_DIR / f'{name}.pdf', bbox_inches='tight'); plt.show()
def find_col(df, kw): return next(c for c in df.columns if kw in c)
unit = lambda col: col.rsplit('-', 1)[-1].upper()

---
## 1. Load data

In [ ]:
bt_runs  = pd.read_csv(RUNS_CSV)
bt_runs  = bt_runs[bt_runs['benchmark'] == BENCHMARK].copy()
bt_means = pd.read_csv(MEAN_CSV)
bt_means = bt_means[bt_means['benchmark'] == BENCHMARK].set_index('language')

COL_CPU_E  = find_col(bt_runs, 'cpu_energy')
COL_MEM_E  = find_col(bt_runs, 'memory_energy')
COL_CARBON = find_col(bt_runs, 'cpu_carbon')
COL_TIME   = find_col(bt_runs, 'phase_time')

print(f'Per-run rows: {len(bt_runs)} | Mean rows: {len(bt_means)}')
bt_means[[COL_CPU_E, COL_TIME]].sort_values(COL_CPU_E).round(3)

---
## 2. Strip plot — per-run consistency

In [ ]:
order = bt_means[COL_CPU_E].sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(8, 6))
sns.stripplot(data=bt_runs, x=COL_CPU_E, y='language', order=order,
              jitter=True, alpha=0.6, size=5, color='#1f77b4', ax=ax)
for i, lang in enumerate(order):
    ax.plot(bt_means.loc[lang, COL_CPU_E], i, marker='D', color='#d62728', markersize=6, zorder=5)

ax.legend(handles=[
    mpatches.Patch(color='#1f77b4', alpha=0.6, label='Individual run (clean)'),
    mpatches.Patch(color='#d62728', label='Mean'),
], loc='lower right')
ax.set_xlabel(f'CPU Energy ({unit(COL_CPU_E)})')
ax.set_ylabel('Language')
ax.set_title(f'{BENCHMARK} — CPU Energy per Run')
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save('stripplot_cpu_energy')

---
## 3. Ranked bar — log scale (span is too large for linear)

In [ ]:
means = bt_means.sort_values(COL_CPU_E)
winner = means.index[0]  # haskell

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

colors = ['#2ca02c'] + ['#1f77b4'] * (len(means) - 1)

# Linear scale
axes[0].barh(means.index, means[COL_CPU_E], color=colors, edgecolor='white')
axes[0].set_xlabel(f'Mean CPU Energy ({unit(COL_CPU_E)})')
axes[0].set_title(f'{BENCHMARK} — Linear scale')
axes[0].xaxis.grid(True, linestyle='--', alpha=0.4)
axes[0].set_axisbelow(True)

# Log scale — better shows the full spread
axes[1].barh(means.index, means[COL_CPU_E], color=colors, edgecolor='white')
axes[1].set_xscale('log')
axes[1].set_xlabel(f'Mean CPU Energy ({unit(COL_CPU_E)}) — log scale')
axes[1].set_title(f'{BENCHMARK} — Log scale (reveals Haskell gap)')
axes[1].xaxis.grid(True, linestyle='--', alpha=0.4)
axes[1].set_axisbelow(True)

plt.suptitle(f'{BENCHMARK} — Haskell is {means.loc["cpp", COL_CPU_E]/means.loc[winner, COL_CPU_E]:.0f}× more efficient than C++', y=1.01)
plt.tight_layout()
save('bar_cpu_energy_linear_log')

---
## 4. Zoom — compiled languages only (exclude interpreted)

In [ ]:
# Interpreted languages distort the chart — show compiled group separately
compiled = ['c', 'cpp', 'rust', 'swift', 'csharp', 'go', 'nodejs', 'fsharp', 'java', 'dart', 'ocaml', 'haskell']
means_comp = means[means.index.isin(compiled)]

fig, ax = plt.subplots(figsize=(6, 5))
colors_comp = ['#2ca02c' if l == winner else '#1f77b4' for l in means_comp.index]
ax.barh(means_comp.index, means_comp[COL_CPU_E], color=colors_comp, edgecolor='white')
ax.set_xlabel(f'Mean CPU Energy ({unit(COL_CPU_E)})')
ax.set_title(f'{BENCHMARK} — Compiled/JIT languages only')
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save('bar_compiled_only')

---
## 5. Normalised to winner

In [ ]:
subset = means[[COL_CPU_E, COL_TIME, COL_MEM_E]]
ratio = subset.div(subset.loc[winner])
ratio.columns = ['CPU Energy', 'Time', 'Mem Energy']

fig, ax = plt.subplots(figsize=(9, 6))
ratio.plot(kind='barh', ax=ax, edgecolor='white')
ax.axvline(1.0, color='black', linewidth=1.2, linestyle='--', label=f'{winner} baseline')
ax.set_xscale('log')
ax.set_xlabel(f'Ratio relative to {winner}  (log scale, lower = better)')
ax.set_title(f'{BENCHMARK} — All Metrics Normalised to {winner} (log scale)')
ax.legend(loc='lower right')
ax.xaxis.grid(True, linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
save('normalised_log')

---
## 6. Energy vs. Time scatter

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for lang, row in means.iterrows():
    color = '#2ca02c' if lang == winner else '#1f77b4'
    ax.scatter(row[COL_TIME], row[COL_CPU_E], color=color, s=80 if lang == winner else 45, zorder=3)
    ax.annotate(lang, (row[COL_TIME], row[COL_CPU_E]),
                textcoords='offset points', xytext=(5, 3), fontsize=8,
                color='#2ca02c' if lang == winner else 'black',
                fontweight='bold' if lang == winner else 'normal')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(f'Time ({unit(COL_TIME)}) — log scale')
ax.set_ylabel(f'CPU Energy ({unit(COL_CPU_E)}) — log scale')
ax.set_title(f'{BENCHMARK} — Energy vs. Time (log–log)')
ax.grid(True, linestyle='--', alpha=0.3, which='both')
plt.tight_layout()
save('scatter_energy_vs_time_loglog')

---
## 7. Summary table

In [ ]:
summary = means[[COL_CPU_E, COL_TIME, COL_MEM_E, COL_CARBON]].copy()
summary.columns = [f'CPU Energy ({unit(COL_CPU_E)})', f'Time ({unit(COL_TIME)})',
                   f'Mem Energy ({unit(COL_MEM_E)})', f'CPU Carbon ({unit(COL_CARBON)})']
summary.round(4)